# 三味線譜変換ツール

五線譜（PDF）→ MusicXML（oemer）→ 三味線中間XML の変換を行います。

**実行順序:** セル0（セットアップ）→ セル1（PDF変換）→ セル2（調弦選択）→ セル3（変換）→ セル4（ダウンロード）

In [ ]:
# ============================================================
# セル0: セットアップ（最初に一度だけ実行）
# ============================================================

REPO_URL = "https://github.com/YOUR_USERNAME/shamisen-converter.git"  # ← GitHubのURLに変更

import os
import sys

REPO_DIR = "shamisen-converter"

if not os.path.exists(REPO_DIR):
    os.system(f"git clone {REPO_URL}")

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, os.getcwd())

print("📦 ライブラリをインストール中...")
os.system("apt-get install -y poppler-utils -q")
os.system("pip install oemer music21 pyyaml pdf2image -q")

print("✅ セットアップ完了")

---
## Step 1: PDF → MusicXML

In [ ]:
# ============================================================
# セル1-a: PDFをアップロード
# ============================================================

from google.colab import files

print("PDFファイルを選択してください")
uploaded = files.upload()

pdf_path = list(uploaded.keys())[0]
print(f"\n📄 アップロード: {pdf_path}")

In [ ]:
# ============================================================
# セル1-b: PDF → 画像 → MusicXML（oemer）
# ============================================================

import os
from pdf2image import convert_from_path
from music21 import converter, stream

IMG_DIR = "_pages"
OMR_DIR = "_omr_output"
os.makedirs(IMG_DIR, exist_ok=True)
os.makedirs(OMR_DIR, exist_ok=True)

# PDF → PNG（ページごと）
print("📃 PDFを画像に変換中...")
pages = convert_from_path(pdf_path, dpi=300)
img_paths = []
for i, page in enumerate(pages):
    img_path = f"{IMG_DIR}/page_{i+1:02d}.png"
    page.save(img_path, "PNG")
    img_paths.append(img_path)
print(f"  → {len(img_paths)} ページ")

# PNG → MusicXML（oemer、ページごと）
print("🎵 oemer で楽譜認識中...")
xml_paths = []
for img_path in img_paths:
    basename = os.path.splitext(os.path.basename(img_path))[0]
    out_dir = f"{OMR_DIR}/{basename}"
    os.makedirs(out_dir, exist_ok=True)
    ret = os.system(f"oemer {img_path} -o {out_dir}")
    # oemer の出力ファイルを探す
    found = [os.path.join(out_dir, f) for f in os.listdir(out_dir) if f.endswith(".xml")]
    if found:
        xml_paths.append(found[0])
        print(f"  ✅ {img_path} → {found[0]}")
    else:
        print(f"  ⚠️  {img_path}: MusicXMLが生成されませんでした")

if not xml_paths:
    raise RuntimeError("MusicXMLが1つも生成されませんでした。PDFの品質を確認してください。")

# 複数ページの場合はマージ
if len(xml_paths) == 1:
    musicxml_path = xml_paths[0]
    print(f"\n📄 MusicXML: {musicxml_path}")
else:
    print("📎 複数ページをマージ中...")
    merged = stream.Score()
    for xp in xml_paths:
        sc = converter.parse(xp)
        for part in sc.parts:
            merged.append(part)
    musicxml_path = f"{OMR_DIR}/merged.xml"
    merged.write("musicxml", fp=musicxml_path)
    print(f"  → {musicxml_path}")

print("\n✅ MusicXML生成完了")

---
## Step 2: 調弦を選択

In [ ]:
# ============================================================
# セル2: 調弦選択
# ============================================================

import ipywidgets as widgets
from IPython.display import display

tuning_widget = widgets.ToggleButtons(
    options=[
        ("本調子", "honchoshi"),
        ("二上り", "niagari"),
        ("三下り", "sansagari"),
    ],
    description="調弦:",
    button_style="info",
)
display(tuning_widget)

---
## Step 3: MusicXML → 三味線中間XML

In [ ]:
# ============================================================
# セル3-a: 変換実行
# ============================================================

from shamisen_converter import (
    convert_musicxml, resolve_out_of_range,
    load_mapping, build_midi_to_position, to_intermediate_xml
)

tuning = tuning_widget.value
print(f"調弦: {tuning_widget.label} ({tuning})")

result = convert_musicxml(
    musicxml_path=musicxml_path,
    mapping_path="shamisen_mapping.yaml",
    tuning=tuning,
)

total = len([n for n in result.notes if n.note_name != "rest"])
out_of_range = len([n for n in result.notes if n.out_of_range])
print(f"\n音符数: {total}")
print(f"音域外: {out_of_range} 件")
if result.warnings:
    print(f"警告数: {len(result.warnings)} 件")

In [ ]:
# ============================================================
# セル3-b: 音域外の処理
# （音域外がない場合はスキップ可）
# ============================================================

mapping = load_mapping("shamisen_mapping.yaml")
midi_map = build_midi_to_position(mapping, tuning)

out_of_range_notes = [n for n in result.notes if n.out_of_range]

if not out_of_range_notes:
    print("✅ 音域外の音符はありません")
else:
    print(f"⚠️  {len(out_of_range_notes)} 件の音域外音符があります")
    print()

    # 一括処理の選択
    policy_widget = widgets.RadioButtons(
        options=[
            ("すべてオクターブ上げて再変換", "up"),
            ("すべてオクターブ下げて再変換", "down"),
            ("すべて休符として扱う", "skip"),
            ("すべて未解決のまま残す", "keep"),
            ("1音ずつ手動で選択", "manual"),
        ],
        description="処理方針:",
        layout=widgets.Layout(width="400px"),
    )
    apply_btn = widgets.Button(description="適用", button_style="success")

    def apply_policy(btn):
        policy = policy_widget.value
        if policy == "manual":
            # interactive=True で1音ずつ処理
            resolve_out_of_range(result, midi_map, interactive=True)
        else:
            for sn in result.notes:
                if not sn.out_of_range:
                    continue
                if policy == "up":
                    new_midi = sn.midi + 12
                    if new_midi in midi_map:
                        s, p = midi_map[new_midi][0]
                        sn.string, sn.position, sn.midi = s, p, new_midi
                        sn.note_name += "(+8va)"
                        sn.out_of_range = False
                        sn.warning = f"オクターブ上げて解決: {sn.note_name}"
                elif policy == "down":
                    new_midi = sn.midi - 12
                    if new_midi in midi_map:
                        s, p = midi_map[new_midi][0]
                        sn.string, sn.position, sn.midi = s, p, new_midi
                        sn.note_name += "(-8va)"
                        sn.out_of_range = False
                        sn.warning = f"オクターブ下げて解決: {sn.note_name}"
                elif policy == "skip":
                    sn.note_name = "rest"
                    sn.out_of_range = False
                    sn.warning = "音域外のためスキップ"
                # keep: そのまま
        print("✅ 音域外処理完了")

    apply_btn.on_click(apply_policy)
    display(policy_widget, apply_btn)

In [ ]:
# ============================================================
# セル3-c: 中間XML出力
# ============================================================

OUTPUT_PATH = "shamisen_output.xml"

xml_str = to_intermediate_xml(result)
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    f.write(xml_str)

print(f"✅ 変換完了 → {OUTPUT_PATH}")

# 先頭30行をプレビュー
lines = xml_str.splitlines()
print("\n--- プレビュー（先頭30行）---")
print("\n".join(lines[:30]))

---
## Step 4: 中間XMLをダウンロード

In [ ]:
# ============================================================
# セル4: ダウンロード
# ============================================================

from google.colab import files
files.download(OUTPUT_PATH)
print(f"⬇️  {OUTPUT_PATH} をダウンロードしました")

---
## （オプション）MusicXMLを直接アップロードして変換

PDFを使わずMusicXMLファイルから始める場合はこちら。

In [ ]:
# ============================================================
# オプション: MusicXMLを直接アップロード
# ============================================================

from google.colab import files

print("MusicXMLファイル（.xml / .mxl）を選択してください")
uploaded_xml = files.upload()
musicxml_path = list(uploaded_xml.keys())[0]
print(f"\n📄 アップロード: {musicxml_path}")
print("→ セル2（調弦選択）から続けてください")